In [1]:
import sys
sys.path.append("../../")
sys.path.append("../../outboxml")

In [2]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import json

In [3]:
from outboxml.extractors import Extractor

In [4]:
class DataExtractor(Extractor):
    def __init__(self):
        super().__init__()
    def extract_dataset(self):
        housing = fetch_california_housing(as_frame=True)
        return pd.concat([housing['data'], housing['target']],axis=1)

In [5]:
from outboxml.automl_utils import build_default_all_models_config

In [6]:
model_params =  {'objective': 'gamma', 'wrapper': 'glm', 'name': 'price', 'colum}
features_params ={'encoding_num': 'WoE_num_to_cat'}

In [7]:
from outboxml.automl_manager import AutoMLManager

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
all_models_config_name = 'configs/house_pricing.json'

In [9]:
all_models_config = build_default_all_models_config(data=DataExtractor().extract_dataset(),
                                                    target_name='MedHouseVal', 
                                                    group_name='example',
                                                    project='house_pricing',
                                                    model_params = model_params,
                                                    features_params=features_params,
                                                   ) 
with open(all_models_config_name, 'w') as f:
    json.dump(dict(all_models_config.model_dump()), f)

2025-12-19 10:16:28.068 | INFO     | outboxml.core.config_builders:feature_params:170 - Prepare feature||MedInc
2025-12-19 10:16:28.073 | INFO     | outboxml.core.config_builders:feature_params:206 - {'type': 'numerical', 'name': 'MedInc', 'clip': {'min_value': 0.536, 'max_value': 15.0}, 'default': '_MEDIAN_', 'encoding': 'WoE_num_to_cat'}
2025-12-19 10:16:28.073 | DEBUG    | outboxml.core.config_builders:build:75 - Feature builder||MedInc
2025-12-19 10:16:28.074 | INFO     | outboxml.core.config_builders:feature_params:170 - Prepare feature||HouseAge
2025-12-19 10:16:28.079 | INFO     | outboxml.core.config_builders:feature_params:206 - {'type': 'numerical', 'name': 'HouseAge', 'clip': {'min_value': 2.0, 'max_value': 52.0}, 'default': '_MEDIAN_', 'encoding': 'WoE_num_to_cat'}
2025-12-19 10:16:28.080 | DEBUG    | outboxml.core.config_builders:build:75 - Feature builder||HouseAge
2025-12-19 10:16:28.081 | INFO     | outboxml.core.config_builders:feature_params:170 - Prepare feature||Ave

In [10]:
all_models_config_name

'configs/house_pricing.json'

In [11]:
auto_ml = AutoMLManager(auto_ml_config='configs/automl-house_pricing.json',
                        models_config=all_models_config_name,
                        extractor=DataExtractor(),
                        retro=False,
                        hp_tune=False)

2025-12-19 10:16:28.141 | DEBUG    | outboxml.datasets_manager:_init_dsmanager:563 - Initializing DSManager
2025-12-19 10:16:28.144 | INFO     | outboxml.datasets_manager:__load_all_models_config:473 - All models config from path
2025-12-19 10:16:28.151 | WARNING  | outboxml.datasets_manager:__load_all_models_config:494 - price||File /home/jovyan/work/results/price_v1_subset.pickle already exists. Change version in config file to for new data prepare
2025-12-19 10:16:28.152 | INFO     | outboxml.datasets_manager:__load_all_models_config:501 - Config is loaded
2025-12-19 10:16:28.153 | INFO     | outboxml.datasets_manager:__load_prepare_datasets:520 - Load models prepare datasets
2025-12-19 10:16:28.154 | INFO     | outboxml.datasets_manager:_init_dsmanager:571 - Reading user extractor
2025-12-19 10:16:28.155 | DEBUG    | outboxml.datasets_manager:_init_dsmanager:579 - Initializing completed
2025-12-19 10:16:28.156 | INFO     | outboxml.automl_manager:__init_auto_ml:459 - All models con

In [12]:
auto_ml.update_models(send_mail=False)

2025-12-19 10:16:28.243 | DEBUG    | outboxml.datasets_manager:fit_models:314 - Fitting model started
2025-12-19 10:16:28.244 | INFO     | outboxml.datasets_manager:fit_models:322 - Setting default models
2025-12-19 10:16:28.257 | INFO     | outboxml.data_subsets:dataset:249 - ['AveBedrms', 'HouseAge', 'MedInc', 'Latitude', 'AveRooms', None, 'Population', 'Longitude', 'AveOccup']
2025-12-19 10:16:28.258 | INFO     | outboxml.data_subsets:dataset:250 -        MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0      8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1      8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2      7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3      5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4      3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   
...       ...       ...       ...        ...         ...       .

SMTPServerDisconnected: please run connect() first

In [ ]:
from outboxml.export_results import ResultExport

In [ ]:
ResultExport(auto_ml).plots(model_name='price', features=['MedInc'], )

In [ ]:
ResultExport(auto_ml).plots(model_name='price', features=['HouseAge'], )

In [ ]:
ResultExport(auto_ml).plots(model_name='price', features=['AveOccup'], )